# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mubashir-dev751/starter/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

Primary Action: Human Review for Content Refresh
The model ranks pages by their probability of being in a state of decline. The output is a prioritized queue for the content strategy team.

Reason Codes (Mapping Features to Context):
To make the probability score actionable, the model flags the most likely contributing factor based on the inputs:

RC1 - Stale Content: days_since_last_update exceeds historical client averages. Action: Evaluate for factual updates.

RC2 - Search Visibility Drop: clean_avg_position is high (poor rank) or has_no_position_data = 1. Action: SEO and intent audit.

RC3 - Poor UX/Engagement: scroll_rate or engagement_rate are in the bottom quartile for the client. Action: Evaluate readability and page layout.

RC4 - General Pattern: No single feature dominates, but the aggregate profile mirrors historically declining pages.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

Intended Use:
This model provides directional decision-support. It surfaces a ranked list of URLs that share characteristics with pages that historically declined in traffic. It is designed solely to help content teams prioritize which pages to audit first.

Known Limits and Blind Spots:

No Causal Authority: The model identifies associations (e.g., stale pages are associated with decline), it does not prove that updating the page will reverse the trend.

Weak Signal: The model achieves a ROC-AUC of 0.601, meaning false positives are highly prevalent.

Target Leakage Context: Features like impressions_90d overlap with the target label's time window. This means the model's current ranking power is partially derived from observing the outcome, inflating its perceived accuracy.

Missing Data Traps: avg_position frequently contains zeros representing missing data, not position zero. While flagged with has_no_position_data, structural missingness across different content_type categories limits cross-category comparisons.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Human Review Rules:
Every URL flagged in the top 20% of the ranked queue must undergo a human audit before any resources are spent updating it. The human reviewer must validate the reason code against current Google Search Console and GA4 data to confirm the decline is real and not a data artifact.

The No-Go List (Strictly Prohibited Actions):

Zero Automated Deletions: The model may never trigger automated archiving, unpublishing, or deletion of a page.

Zero Automated Redirects: The model may not automate 301 redirects to other pages.

No Performance Punishments: Model scores must not be used to evaluate the performance of the writers or SEOs who created the content.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

Monitoring:

Track the distribution of decline_probability. If the model begins scoring a disproportionate majority of pages above 0.80 or below 0.20, the input data distribution has shifted.

Monitor has_no_position_data ratios. If the upstream data pipeline breaks and this feature spikes, the model's outputs become instantly invalid.

Retrain Triggers:

Methodological Fix: The immediate trigger for retraining is resolving the time-window leakage between the 90-day input features and the 30-day target label.

Performance Floor: If a future honest holdout drops below a 0.55 ROC-AUC, the model is retired until new features are engineered.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [6]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

os.makedirs('work/outputs', exist_ok=True)
os.makedirs('work/figures', exist_ok=True)

url = 'https://raw.githubusercontent.com/mubashir-dev751/starter/main/data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(url)

df['is_declining_label'] = (df['trend_direction'].str.lower() == 'down').astype(int)
df['has_no_position_data'] = (df['avg_position'] == 0).astype(int)
df['clean_avg_position'] = np.where(df['avg_position'] == 0, np.nan, df['avg_position'])
df['has_missing_word_count'] = df['word_count'].isna().astype(int)

numeric_features = ['impressions_90d', 'clicks_90d', 'ctr', 'clean_avg_position',
                    'has_no_position_data', 'days_since_last_update', 'has_missing_word_count',
                    'engagement_rate', 'scroll_rate', 'ai_traffic_pct']
categorical_features = ['content_type']
features = numeric_features + categorical_features
target = 'is_declining_label'

preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_features),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='missing')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_features)
])
model = Pipeline([
    ('prep', preprocessor),
    ('rf', RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1))
])

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['client_id']))
model.fit(df.iloc[train_idx][features], df.iloc[train_idx][target])

df['decline_probability'] = model.predict_proba(df[features])[:, 1]

def assign_reason_code(row):
    if row['days_since_last_update'] > 365:
        return 'RC1: Stale Content (>365 days)'
    elif row['has_no_position_data'] == 1 or row['clean_avg_position'] > 50:
        return 'RC2: Poor/Missing Search Visibility'
    elif row['engagement_rate'] < 30:
        return 'RC3: Low Engagement Rate'
    else:
        return 'RC4: General Profile Match'

df['reason_code'] = df.apply(assign_reason_code, axis=1)

export_cols = ['client_id', 'content_id', 'decline_probability', 'reason_code']
ranked_queue = df[export_cols].sort_values(by='decline_probability', ascending=False)

ranked_queue.to_csv('work/outputs/ranked_queue_w07.csv', index=False)

print(f"Exported ranked queue with {len(ranked_queue)} rows to work/outputs/ranked_queue_w07.csv")
print("\nTop 5 recommended actions:")
print(ranked_queue.head(5))

Exported ranked queue with 30000 rows to work/outputs/ranked_queue_w07.csv

Top 5 recommended actions:
               client_id            content_id  decline_probability  \
18406  client_7f2253d7e2  content_d75823fc91dc             0.830066   
26532  client_7f2253d7e2  content_370de6e8e035             0.828734   
10742  client_7f2253d7e2  content_148abe09b846             0.827996   
18646  client_7f2253d7e2  content_acf71f98ada2             0.827332   
29921  client_7f2253d7e2  content_c2ac8518bb1f             0.825720   

                    reason_code  
18406  RC3: Low Engagement Rate  
26532  RC3: Low Engagement Rate  
10742  RC3: Low Engagement Rate  
18646  RC3: Low Engagement Rate  
29921  RC3: Low Engagement Rate  


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.